# Día 9 · Unidad 10 — Estadística aplicada (III): pruebas de hipótesis con SciPy

**Objetivos de aprendizaje**
- Ejecutar un t-test de una muestra (`ttest_1samp`) y de dos muestras independientes (`ttest_ind`) con `scipy.stats`.
- Ejecutar una prueba de independencia chi-cuadrado (`chi2_contingency`) sobre una tabla de contingencia.
- Interpretar correctamente un p-valor: qué afirma y, sobre todo, qué **no** afirma.
- Usar la IA para traducir un resultado estadístico a lenguaje de negocio, verificando siempre la traducción contra el propio cálculo numérico.

**Duración estimada:** 90-100 minutos.

**Requisitos previos:** `05_teoria_distribuciones_probabilidad.ipynb` de este mismo día. Igual que en los dos notebooks anteriores de la unidad, no se explica desde cero qué es una hipótesis nula o un p-valor — el foco es la sintaxis de `scipy.stats` y, sobre todo, no dar por buena una interpretación sin comprobarla.


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats

cartera = pd.read_csv("../data/raw/cartera_polizas.csv")
siniestros = pd.read_csv("../data/raw/siniestros.csv")

ALFA = 0.05  # nivel de significación que usaremos en todo el notebook


## 1. t-test de una muestra: `ttest_1samp`

**Caso de negocio**: el equipo de tarificación de AUTO fijó hace tiempo una severidad media de referencia de **1.500 €** por siniestro para sus cálculos de prima pura. ¿La severidad media observada en la cartera actual es significativamente distinta de esa referencia, o la diferencia observada es compatible con variación muestral?

- H₀ (hipótesis nula): la media poblacional de severidad de AUTO es 1.500 €.
- H₁ (hipótesis alternativa): la media poblacional de severidad de AUTO **no** es 1.500 €.

`scipy.stats.ttest_1samp(datos, popmean)` devuelve el estadístico t y el p-valor de un contraste bilateral (dos colas) por defecto.


In [ ]:
severidad_auto = siniestros.loc[siniestros["producto"] == "AUTO", "importe_pagado"].to_numpy()
referencia_tarificacion = 1500.0

print(f"n = {len(severidad_auto)}")
print(f"Media observada: {severidad_auto.mean():.2f} €")
print(f"Desv. estándar observada: {severidad_auto.std(ddof=1):.2f} €")

resultado_1samp = scipy.stats.ttest_1samp(severidad_auto, popmean=referencia_tarificacion)
print(f"\nEstadístico t = {resultado_1samp.statistic:.3f}")
print(f"p-valor       = {resultado_1samp.pvalue:.4f}")


In [ ]:
if resultado_1samp.pvalue < ALFA:
    print(f"p-valor ({resultado_1samp.pvalue:.4f}) < alfa ({ALFA}): se RECHAZA H0.")
    print("La severidad media observada es estadísticamente distinta de los 1.500 € de referencia.")
else:
    print(f"p-valor ({resultado_1samp.pvalue:.4f}) >= alfa ({ALFA}): NO se rechaza H0.")
    print("La diferencia frente a los 1.500 € de referencia es compatible con variación muestral")
    print("(no hay evidencia estadística suficiente para afirmar que la media poblacional es distinta).")


> ⚠️ **Error típico**: interpretar "no se rechaza H₀" como "H₀ es cierta" o "se ha demostrado que la media es 1.500 €". Un test de hipótesis nunca demuestra la hipótesis nula — solo dice si hay evidencia suficiente en la muestra para **rechazarla**. "No rechazar" es una conclusión más débil que "aceptar": significa "los datos no dan evidencia suficiente para descartar H₀", no "los datos confirman H₀".

**Intervalo de confianza como complemento**: el p-valor solo dice "significativo o no" a un nivel dado; el intervalo de confianza os da además una idea de la magnitud plausible de la media real, que suele ser más útil para una decisión de negocio.


In [ ]:
# Intervalo de confianza al 95% para la media poblacional (usando la distribución t de Student)
media_muestral = severidad_auto.mean()
error_estandar = severidad_auto.std(ddof=1) / np.sqrt(len(severidad_auto))
gl = len(severidad_auto) - 1  # grados de libertad

ic_95 = scipy.stats.t.interval(confidence=0.95, df=gl, loc=media_muestral, scale=error_estandar)
print(f"Media muestral: {media_muestral:.2f} €")
print(f"IC 95%: [{ic_95[0]:.2f} €, {ic_95[1]:.2f} €]")
print(f"¿La referencia de 1.500 € cae dentro del IC 95%?: {ic_95[0] <= referencia_tarificacion <= ic_95[1]}")
print("(esta comprobación es equivalente al test de hipótesis a alfa=0.05: si la referencia cae")
print(" DENTRO del IC, el test NO rechaza H0; si cae FUERA, el test SÍ la rechaza)")


**Para ti:** repite el mismo contraste, pero para el producto `HOGAR` con la misma referencia de 1.500 €. ¿Cambia la conclusión respecto a AUTO?


In [ ]:
# Tu código aquí


## 2. t-test de dos muestras independientes: `ttest_ind`

**Caso de negocio**: ¿la severidad media de los siniestros de AUTO es significativamente distinta de la de HOGAR? Es una pregunta habitual al revisar si la tarificación por producto está bien diferenciada.

- H₀: las medias poblacionales de severidad de AUTO y HOGAR son iguales.
- H₁: son distintas.

Por defecto, `ttest_ind` asume varianzas iguales entre los dos grupos (`equal_var=True`) — un supuesto que conviene comprobar, no dar por hecho. Si las varianzas muestrales son claramente distintas (como suele pasar con severidad de siniestros, muy dispersa), es más prudente usar el **test de Welch** (`equal_var=False`), que no asume varianzas iguales.


In [ ]:
severidad_hogar = siniestros.loc[siniestros["producto"] == "HOGAR", "importe_pagado"].to_numpy()

print(f"AUTO  -- n={len(severidad_auto):>4}, media={severidad_auto.mean():>9.2f} €, "
      f"desv.est={severidad_auto.std(ddof=1):>9.2f} €")
print(f"HOGAR -- n={len(severidad_hogar):>4}, media={severidad_hogar.mean():>9.2f} €, "
      f"desv.est={severidad_hogar.std(ddof=1):>9.2f} €")

# Comprobación del supuesto de varianzas iguales antes de elegir equal_var
razon_varianzas = severidad_auto.var(ddof=1) / severidad_hogar.var(ddof=1)
print(f"\nRatio de varianzas AUTO/HOGAR: {razon_varianzas:.2f} "
      "(una regla práctica habitual: si está fuera de [0.5, 2], mejor no asumir varianzas iguales)")


In [ ]:
resultado_welch = scipy.stats.ttest_ind(severidad_auto, severidad_hogar, equal_var=False)
print(f"t (Welch) = {resultado_welch.statistic:.3f}, p-valor = {resultado_welch.pvalue:.4f}")

if resultado_welch.pvalue < ALFA:
    print(f"\np-valor < {ALFA}: se rechaza H0 -- hay diferencia estadísticamente significativa")
    print("entre la severidad media de AUTO y la de HOGAR.")
else:
    print(f"\np-valor >= {ALFA}: no se rechaza H0 -- no hay evidencia suficiente de que la severidad")
    print("media de AUTO y HOGAR sean distintas en la población.")


> ⚠️ **Error típico**: usar `equal_var=True` (el valor por defecto de `ttest_ind`) sin comprobar antes si las varianzas de los dos grupos son parecidas. Con varianzas muy distintas, el t-test de Student clásico puede dar un p-valor poco fiable (demasiado optimista o pesimista según el caso); el test de Welch (`equal_var=False`) es la opción más robusta por defecto salvo que tengáis buenas razones para asumir varianzas iguales — de hecho, algunas guías estadísticas recomiendan usar Welch siempre, como opción segura por defecto.


## 3. Chi-cuadrado de independencia: `chi2_contingency`

**Caso de negocio**: ¿el **producto** de la póliza está relacionado con que el siniestro asociado termine **abierto** o **cerrado**? Si hubiera una asociación fuerte (p. ej. los siniestros de un producto tardan sistemáticamente más en cerrarse), sería relevante para gestión operativa.

Es una prueba sobre dos variables **categóricas** — no numéricas como en los t-tests anteriores — así que primero se construye una tabla de contingencia (recordáis `pivot_table`/`crosstab` del Día 8).

- H₀: `producto` y `estado` son independientes (no hay asociación).
- H₁: `producto` y `estado` están asociados.


In [ ]:
tabla_contingencia = pd.crosstab(siniestros["producto"], siniestros["estado"])
tabla_contingencia


In [ ]:
chi2, p_valor_chi2, gl_chi2, frecuencias_esperadas = scipy.stats.chi2_contingency(tabla_contingencia)

print(f"Estadístico chi2 = {chi2:.3f}")
print(f"Grados de libertad = {gl_chi2}")
print(f"p-valor = {p_valor_chi2:.4f}")
print("\nFrecuencias esperadas bajo independencia (H0):")
pd.DataFrame(frecuencias_esperadas, index=tabla_contingencia.index, columns=tabla_contingencia.columns).round(1)


In [ ]:
if p_valor_chi2 < ALFA:
    print(f"p-valor ({p_valor_chi2:.4f}) < alfa: se rechaza H0 -- producto y estado del siniestro")
    print("no son independientes en esta muestra (hay asociación estadísticamente significativa).")
else:
    print(f"p-valor ({p_valor_chi2:.4f}) >= alfa: no se rechaza H0 -- no hay evidencia suficiente")
    print("de asociación entre producto y estado del siniestro en esta muestra.")


> ⚠️ **Error típico**: aplicar chi-cuadrado con frecuencias esperadas muy bajas (una regla práctica común: más de un 20% de las celdas con frecuencia esperada por debajo de 5). El test asume un tamaño de muestra razonable en cada celda; con conteos muy pequeños el p-valor deja de ser fiable, y conviene revisar `frecuencias_esperadas` (como hicimos arriba) antes de confiar en el resultado — o agrupar categorías con pocos casos.

**Para ti:** construye la tabla de contingencia entre `producto` y `tipo_siniestro` (cuidado: `tipo_siniestro` depende del producto, así que muchas celdas serán 0 por diseño — esto es justo el tipo de caso donde conviene revisar las frecuencias esperadas antes de interpretar el resultado). Ejecuta `chi2_contingency` e inspecciona si hay celdas con frecuencia esperada baja.


In [ ]:
# Tu código aquí


## 4. Con ayuda de la IA: traducir un resultado a lenguaje de negocio (validando siempre)

Un actuario rara vez entrega un p-valor en crudo a un comité de dirección — hay que traducirlo a una frase clara, sin tecnicismos, y **sin perder rigor por el camino**. Es exactamente el tipo de tarea mecánica y repetitiva donde un asistente de IA ayuda mucho — con la condición de comprobar siempre la traducción contra el cálculo numérico real (`docs/uso_ia_en_el_curso.md`: "explicar resultados estadísticos en lenguaje de negocio, validando siempre numéricamente").


In [ ]:
prompt_traduccion = f"""
Contexto: soy actuario/a y acabo de ejecutar un t-test de dos muestras independientes
(test de Welch) en Python con scipy.stats para comparar la severidad media de los
siniestros de AUTO frente a los de HOGAR.

Resultado exacto obtenido:
- Severidad media AUTO: {severidad_auto.mean():.2f} EUR (n={len(severidad_auto)})
- Severidad media HOGAR: {severidad_hogar.mean():.2f} EUR (n={len(severidad_hogar)})
- Estadistico t = {resultado_welch.statistic:.3f}
- p-valor = {resultado_welch.pvalue:.4f}
- Nivel de significacion usado: alfa = 0.05

Objetivo: traduce este resultado a una frase (maximo 3 lineas) para un informe
dirigido a un comite de direccion sin formacion estadistica, que sea correcta,
sin sobre-interpretar el resultado (no digas "la diferencia es grande" si el test
no dice nada sobre el tamaño practico, solo sobre significacion estadistica).

Restricciones: no uses la palabra "p-valor" ni "hipotesis nula" en la frase final
(el comite no esta familiarizado con esos terminos), pero el contenido debe seguir
siendo estadisticamente preciso. Indica tambien, en una linea aparte, el valor
numerico exacto del p-valor para que quede trazable en el informe.

Formato de salida: la frase para el informe, mas la linea con el p-valor exacto.
"""
print(prompt_traduccion)


**Qué se espera obtener**
- Una frase en lenguaje llano que indique si la diferencia observada es (o no) "estadísticamente significativa" traducido a términos de negocio, del tipo *"con los datos disponibles, no hay evidencia suficiente para afirmar que la severidad media de AUTO y HOGAR sean distintas"* (o la afirmación contraria, según el resultado real).
- Que no confunda "no significativo" con "las medias son iguales" (el mismo matiz que remarcamos en el apartado 1).
- El p-valor exacto citado en la respuesta, para que el informe quede trazable.

**Qué debes verificar antes de darlo por bueno**
- Que la dirección de la conclusión (significativo / no significativo) **coincide exactamente** con `resultado_welch.pvalue < ALFA` calculado arriba — no te fíes de que "suene razonable", compára el número.
- Que el p-valor citado en la frase coincide, cifra a cifra, con `resultado_welch.pvalue` — un LLM puede "redondear mal" o inventar un número parecido pero no idéntico (alucinación numérica, ver `docs/uso_ia_en_el_curso.md`).
- Que la frase **no** afirma nada sobre la magnitud práctica de la diferencia (cuántos euros de diferencia, si es relevante para el negocio) a partir únicamente del resultado del test — significación estadística y relevancia práctica son cosas distintas, y con muestras grandes hasta una diferencia pequeña en euros puede salir "significativa".

*(Verificación programática de la primera comprobación — nunca os fiéis solo de la lectura visual):*


In [ ]:
es_significativo = resultado_welch.pvalue < ALFA
frase_esperada = (
    "SÍ hay diferencia estadísticamente significativa"
    if es_significativo
    else "NO hay evidencia suficiente de diferencia significativa"
)
print(f"Verificación: {frase_esperada} entre la severidad de AUTO y HOGAR (p={resultado_welch.pvalue:.4f}).")
print("Cualquier traducción de la IA que diga lo contrario está mal y no se acepta, por bien que 'suene'.")


## Resumen

- `ttest_1samp(datos, popmean)`: contraste de una muestra contra un valor de referencia. `ttest_ind(x, y, equal_var=...)`: contraste de dos muestras independientes — comprobad el supuesto de varianzas iguales antes de fijar `equal_var`, y usad Welch (`equal_var=False`) como opción robusta por defecto.
- `chi2_contingency(tabla)`: prueba de independencia entre dos variables categóricas a partir de una tabla de contingencia (`pd.crosstab`) — revisad las frecuencias esperadas devueltas antes de confiar en el resultado si algún grupo es pequeño.
- "No rechazar H₀" **no** equivale a "demostrar H₀" — solo significa que la muestra no da evidencia suficiente para descartarla.
- Un intervalo de confianza (`scipy.stats.t.interval(...)`) complementa al p-valor con una idea de magnitud, no solo de "significativo sí/no".
- Significación estadística y relevancia práctica (de negocio) son conceptos distintos: con muestras grandes, hasta diferencias pequeñas pueden salir "significativas" sin ser relevantes para una decisión.
- Un asistente de IA puede traducir un resultado estadístico a lenguaje de negocio muy bien y muy rápido — pero la traducción se acepta solo después de comprobar, cifra a cifra, que la dirección de la conclusión y el p-valor citado coinciden exactamente con vuestro propio cálculo.

**Siguiente:** `07_ejercicios.ipynb`, que combina fuentes de datos (BBDD/API) y estadística aplicada en ejercicios de dificultad creciente.
